In [8]:
import os
import json
from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain.vectorstores import FAISS
from langchain.schema.output_parser import StrOutputParser
from langchain_groq import ChatGroq
from langchain.schema.runnable import RunnablePassthrough

In [11]:
os.environ["GROQ_API_KEY"]= json.loads(os.getenv("API_KEYS"))['GROQ_API_KEY']

In [12]:
#pdf_url= "https://s1.q4cdn.com/806093406/files/doc_financials/2025/q4/Q4-FY25_Press-Release_FINAL.pdf"
loader= PyPDFLoader(r"C:\Users\chakr\Downloads\projects\deploy_rag\data\1706.03762v7.pdf")
data= loader.load()

In [13]:
text_splitter= RecursiveCharacterTextSplitter(chunk_size= 200, chunk_overlap= 50)
text_chunks= text_splitter.split_documents(data)

In [14]:
embeddings=HuggingFaceEmbeddings(model_name= "BAAI/bge-small-en")

In [15]:
vectorstore=FAISS.from_documents(text_chunks, embeddings)

In [16]:
retriever=vectorstore.as_retriever()

In [19]:
query = "What is attention ?"
docs = vectorstore.similarity_search(query, k=4)

# Display the results
for i, doc in enumerate(docs):
    print(f"Document {i+1}:")
    print(doc.page_content)
    print("-" * 50)

Document 1:
3.2 Attention
An attention function can be described as mapping a query and a set of key-value pairs to an output,
--------------------------------------------------
Document 2:
attention and the parameter-free position representation and became the other person involved in nearly every
--------------------------------------------------
Document 3:
values.
In practice, we compute the attention function on a set of queries simultaneously, packed together
--------------------------------------------------
Document 4:
to investigate local, restricted attention mechanisms to efficiently handle large inputs and outputs
--------------------------------------------------


In [20]:
from langchain.prompts import ChatPromptTemplate

template="""You are an assistant for question-answering tasks.
Use the following pieces of retrieved context to answer the question.
If you don't know the answer, just say that you don't know.
Use ten sentences maximum and keep the answer concise.
Question: {question}
Context: {context}
Answer:
"""

In [21]:
prompt=ChatPromptTemplate.from_template(template)

In [22]:
output_parser=StrOutputParser()

In [23]:
llm= ChatGroq(model= "openai/gpt-oss-20b")

In [24]:
rag_chain = (
    {"context": retriever,  "question": RunnablePassthrough()}
    | prompt
    | llm
    | output_parser
)

In [25]:
rag_chain.invoke("What is attention?")

'Attention is a mechanism that transforms a query together with a set of key‑value pairs into an output. It computes a weighted sum of the values, where the weights are derived from the similarity between the query and each key. In practice, many queries are processed simultaneously, allowing the computation to be batched efficiently. Attention can be local or restricted to reduce computational cost for very long inputs. The core idea is that the output focuses on the most relevant parts of the input, as indicated by the key‑query match. This mapping is parameter‑free with respect to position, relying only on learned query, key, and value projections. The attention function is a building block of transformer models, enabling them to capture long‑range dependencies. It replaces recurrent or convolutional structures with a simple dot‑product similarity. In short, attention lets a model decide which parts of the input to emphasize when generating each output element.'